# One Health: follow the evidence

**Executed analysis, 9 November 2026.** Start with a question: when the same resistance-marker label appears in several ecological categories, what have we actually learned?

This notebook reproduces the prototype's counts from a pinned Hong Kong plasmid table. It uses standard Python; no account key or model training is needed. The cached-data branch was executed locally. The download branch is provided for Colab but was not executed here.

Data: [Xu et al., 2025](https://doi.org/10.1038/s41467-025-62455-w). These are aquatic E. coli isolates and plasmid records, not individual human patients. This is **not the MetaSUB dataset**.


In [ ]:
from pathlib import Path
import csv, io, hashlib, urllib.request
from collections import Counter

COMMIT = 'dfc043d06d7a1d3783b2424a9303952291095242'
SOURCE_URL = f'https://raw.githubusercontent.com/xxqxxq1994/EC_Ecological-connectivity/{COMMIT}/revused-plasmid-ARG.txt'
EXPECTED_SHA256 = 'bc4e1f22f70209e72a323f6a0318a7350240ffd235e11e50f698d7a748c84cee'
candidates = [Path('data/source.tsv'), Path('../data/source.tsv')]
cached = next((p for p in candidates if p.is_file()), None)
if cached:
    raw = cached.read_bytes()
    origin = 'Local, checksum-verified source cache'
else:
    with urllib.request.urlopen(SOURCE_URL, timeout=30) as response:
        raw = response.read()
    origin = 'Downloaded pinned public source'
# Canonical UTF-8 text: LF newlines and one final newline.
text = raw.decode('utf-8-sig').replace('\r\n', '\n').rstrip('\n') + '\n'
digest = hashlib.sha256(text.encode('utf-8')).hexdigest()
assert digest == EXPECTED_SHA256, 'Unexpected source content: stop and review before using it.'
rows = list(csv.DictReader(io.StringIO(text), delimiter='\t'))
print(origin)
print('Canonical SHA-256:', digest)
print('Rows:', len(rows))


## 1. Establish the unit before counting

Each row is a plasmid record. `Host` identifies an E. coli isolate; an isolate can have several plasmids. `Niche` is ecological attribution. Preserve missing categories rather than quietly dropping them.


In [ ]:
assert len(rows) == 3522
assert len({r['Plasmid-ID'] for r in rows}) == len(rows)
assert len({r['Host'] for r in rows}) == 942
categories = Counter(r['Niche'].strip() or 'Unspecified' for r in rows)
circular = Counter(r['Circular'] for r in rows)
print('Plasmid records:', len(rows))
print('Distinct E. coli isolate IDs:', len({r['Host'] for r in rows}))
print('Ecological categories:', dict(categories))
print('Circular assembly codes:', dict(circular))
print('The paper studies 1,016 isolates; this plasmid table is a different denominator.')


## 2. Ask one exact question

How many plasmid records contain the annotation `tetracycline__tet(A)`? Match complete comma-separated tokens. Repeated annotations within a record count once. These are descriptive record fractions, not population prevalence estimates.


In [ ]:
MARKER = 'tetracycline__tet(A)'  # Try beta_lactam__CTX-M-55 or sulfonamide__sul2.
GROUPS = ['Human', 'Animal', 'Environment']
def tokens(row):
    return {part.strip() for part in row['ARG_Type_Subtype'].split(',') if part.strip()}
def summary(pool, marker):
    result = []
    for group in GROUPS:
        subset = [r for r in pool if r['Niche'] == group]
        found = sum(marker in tokens(r) for r in subset)
        result.append((group, found, len(subset), 100*found/len(subset) if subset else None))
    return result
def show_summary(pool, marker):
    print(f'{"Category":<16} {"Contains marker":>16} {"Records":>10} {"Percent":>10}')
    for group, count, total, pct in summary(pool, marker):
        pct_text = f'{pct:.1f}%' if pct is not None else 'n/a'
        print(f'{group:<16} {count:>16} {total:>10} {pct_text:>10}')
    print('All records:', len(pool), '| marker-positive records:', sum(marker in tokens(r) for r in pool))
    print('Unclassified records retained in overall total:', sum(r['Niche'] not in GROUPS for r in pool))
show_summary(rows, MARKER)
assert sum('tetracycline__tet(A)' in tokens(r) for r in rows) == 815


## 3. Change the recovery method

`Selection=N` means recovered without antibiotic selection in the laboratory. It does not describe someone's antibiotic use. Filtering changes which isolates/plasmids are included and their category mix. Neither view is a random sample of Hong Kong. Do not claim that the difference is a causal effect of the selection procedure.


In [ ]:
without_selection = [r for r in rows if r['Selection'] == 'N']
assert len(without_selection) == 1111
assert sum('tetracycline__tet(A)' in tokens(r) for r in without_selection) == 149
show_summary(without_selection, MARKER)
print('\nQuestion: why is the denominator as important as the coloured bar?')


## 4. Inspect the actual three example records

The examples below are all circular assemblies recovered without antibiotic selection. A matching marker name is not an alignment of full plasmid sequences. `Conj_type` is a predicted mobility category, not the result of a transfer experiment on each row.


In [ ]:
example_ids = ['2ndManhole-barcode01-contig_7', 'pig-inf-barcode52-contig_7', 'RMB-NoR-barcode52-contig_4']
by_id = {r['Plasmid-ID']: r for r in rows}
for record_id in example_ids:
    r = by_id[record_id]
    assert r['Selection'] == 'N' and r['Circular'] == 'Y'
    assert 'tetracycline__tet(A)' in tokens(r)
    print(record_id)
    print('  Source:', r['Source'], '| category:', r['Niche'], '| bp:', r['Length'], '| predicted mobility:', r['Conj_type'])
print('\nWhat we have: a shared annotation in three records.')
print('What we have not established: direct transfer, direction, current exposure, infection risk.')


## 5. Translate evidence into a project decision

- **Education:** test whether a guided explanation helps learners distinguish bacterial spread, gene sharing and what environmental records cannot prove. The existing explorer alone is not a complete transmission lesson.
- **Research support:** find a named scientist who needs a comparison, then audit metadata and calculate only defensible comparisons. A sampler or prediction model needs its own target, labels and validation.
- **MetaSUB next:** use the [official metadata repository](https://github.com/MetaSUB/MetaSUB-metadata) to audit specimen, date, method and available AMR measurement before joining tables. Do not compare cities on raw marker counts. The 2021 atlas and later releases need separate provenance.

The [CHP 2025 public survey](https://www.chp.gov.hk/en/static/109987.html) identifies a gap in knowledge of person-to-person spread of resistant bacteria. Our proposed audience and intervention still require testing. No live surveillance, risk ranking or clinical advice is generated by this notebook.
